# Sacramento Kings International Scouting Report

This notebook is my main report after progress in json.ipynb and report.ipynb of cleaning, merging, and analyzing basketball prospect data.
I now analyze team needs and recommend international players to scout.

Date: November 01, 2025

Team: Sacramento Kings

Goal: Highlight international players outside the NBA for scouting.

In [209]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

In [210]:
merged_df      = pd.read_csv('merged.csv')
kings_merged   = pd.read_csv('2425_kings_merged.csv')
league_teams   = pd.read_csv('2425_league_teams_merged.csv')

In [211]:
def clean(df):
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace('%', '_pct', regex=False))
    return df

kings_merged = clean(kings_merged)
league_teams = clean(league_teams)

## Step 3: Filter International Prospects  
First I identified international players who:  
- Appeared in **at least 10 games per season**  
- Were **under 25 years old** (or age unknown) in 2021  
- Were **not in the NBA in 2021**  
This creates a reliable pool of draft-eligible prospects from 2016–2021.

In [212]:
# Normalize names
merged_df['first_name'] = merged_df['first_name'].astype(str).str.strip().str.title()
merged_df['last_name']  = merged_df['last_name'].astype(str).str.strip().str.title()
merged_df['full_name_norm'] = merged_df['first_name'].str.lower() + '|' + merged_df['last_name'].str.lower()

# Filter seasons
merged_df = merged_df[(merged_df['season'] >= 2010) & (merged_df['season'] <= 2021)]

# # Exclude NBA 2021 players
# nba_2021 = merged_df[(merged_df['league'] == 'NBA') & (merged_df['season'] == 2021)]
# exclude = set(nba_2021['full_name_norm'])

# International prospects
prospects_raw = merged_df[
    (merged_df['league'] != 'NBA') &
    (merged_df['season'] >= 2016) &
    (merged_df['season'] <= 2021) &
    (merged_df['games'] >= 10)
    #   &
    # (~merged_df['full_name_norm'].isin(exclude))
].copy()

print(f"Prospects after filter: {len(prospects_raw)}")

Prospects after filter: 1747


## Step 4: Impute, Aggregate & Compute Per-Game Stats  
Missing values in key counting stats are imputed using **season-level medians** to preserve league context. 

Players are aggregated across seasons, and per-game stats (e.g., `points_pg`, `assists_pg`) are calculated. 

Shooting percentages (`fg_pct`, `3p_pct`, `ft_pct`) and **True Shooting % (TS%)** are derived for efficiency analysis.

In [213]:
key_cols = [
    'points','assists','offensive_rebounds','defensive_rebounds',
    'steals','blocked_shots','turnovers',
    'two_points_made','two_points_attempted',
    'three_points_made','three_points_attempted',
    'free_throws_made','free_throws_attempted'
]

# Impute per season
for col in key_cols:
    medians = prospects_raw.groupby('season')[col].transform('median')
    prospects_raw[col] = prospects_raw[col].fillna(medians)

# Fill team
prospects_raw = prospects_raw.sort_values(['full_name_norm', 'season'])
prospects_raw['team'] = prospects_raw.groupby('full_name_norm')['team'].transform(lambda x: x.ffill().bfill())

# Age
prospects_raw['birth_date'] = pd.to_datetime(prospects_raw['birth_date'], errors='coerce')
prospects_raw['age'] = ((datetime(2021, 11, 1) - prospects_raw['birth_date']).dt.days / 365.25).round(1)
prospects_raw = prospects_raw[(prospects_raw['age'] < 25) | (prospects_raw['age'].isna())].copy()

# Aggregate per player
agg_dict = {c: 'sum' for c in key_cols}
agg_dict.update({
    'games': 'sum',
    'minutes': 'sum',
    'team': 'last',
    'league': 'last',
    'age': 'mean',
    'internal_box_plus_minus': 'mean'
})

prospects_agg = prospects_raw.groupby(['first_name', 'last_name'], as_index=False).agg(agg_dict)

# Per-game
for c in key_cols:
    prospects_agg[f'{c}_pg'] = prospects_agg[c] / prospects_agg['games']

# Percentages
prospects_agg['fg_pct'] = (prospects_agg['two_points_made'] + prospects_agg['three_points_made']) / \
                          (prospects_agg['two_points_attempted'] + prospects_agg['three_points_attempted']).replace(0, np.nan)
prospects_agg['3p_pct'] = prospects_agg['three_points_made'] / prospects_agg['three_points_attempted'].replace(0, np.nan)
prospects_agg['ft_pct'] = prospects_agg['free_throws_made'] / prospects_agg['free_throws_attempted'].replace(0, np.nan)

# TS%
fga = prospects_agg['two_points_attempted'] + prospects_agg['three_points_attempted']
fta = prospects_agg['free_throws_attempted']
# prospects_agg['ts_pct'] = prospects_agg['points_pg'] / (2 * (fga + 0.44 * fta).replace(0, np.nan))
# TS% — use per-game components
prospects_agg['ts_pct'] = (
    prospects_agg['points_pg'] / 
    (2 * (prospects_agg['two_points_attempted_pg'] + prospects_agg['three_points_attempted_pg'] + 0.44 * prospects_agg['free_throws_attempted_pg']))
).replace([np.inf, -np.inf], np.nan).fillna(0.5)

# BPM
prospects_agg['bpm'] = prospects_agg['internal_box_plus_minus']
# After per-game stats
# prospects_agg['bpm'] = prospects_agg['internal_box_plus_minus'] / prospects_agg['games']  # if total

## Step 5: Extract Kings & League Team Averages  
We isolate the **Kings team totals row** (or fall back to roster average) and compute **per-game values** for counting stats. 

The league average is calculated across all teams in `league_teams`. **Box Plus/Minus (BPM)** is included: Kings use player-level average; league uses neutral 0.0.

In [214]:
# Kings team totals
kings_team = kings_merged[kings_merged['player'].str.contains('total|team', case=False, na=False)]
if kings_team.empty:
    kings_team = kings_merged.mean(numeric_only=True).to_frame().T
else:
    kings_team = kings_team.iloc[0]

# League averages (all teams)
league_avg = league_teams.mean(numeric_only=True)

# Per-game
per_game_cols = ['pts','ast','trb','stl','blk','tov','orb','drb']
for col in per_game_cols:
    if col in kings_team.index:
        kings_team[col] = kings_team[col] / kings_team.get('g', 1)
    if col in league_avg.index:
        league_avg[col] = league_avg[col] / league_avg.get('g', 1)

# BPM: Kings player average
kings_bpm = kings_merged['bpm'].mean()
league_bpm = 0.0  # neutral

kings_team['bpm'] = kings_bpm
league_avg['bpm'] = league_bpm

## Step 6: Define Kings Roster Needs (No League Data Required)  
I did calculated based on **ignore league averages**, but we will ignore those to avoid `inf` errors and data mismatches as I did not figure out how to  fix this.

Instead, we define **exactly what the Kings need most** (e.g., rebounding, playmaking, impact via BPM). Each need is assigned a **priority weight** based on team gaps.  
- **BPM gets 20% weight** — overall impact is king.  
- **TOV is penalized** (lower = better).  
- Weights sum to 1.0 → transparent, tunable, and **error-proof**.

In [215]:
# --------------------------------------------------------------
# 6. KINGS WEAKNESSES — SAFE, NO ZERO, NO INF
# --------------------------------------------------------------

metrics = ['pts','ast','trb','stl','blk','orb','drb','tov','fg_pct','3p_pct','ft_pct','ts_pct','bpm']

# Kings team row
kings_team_row = kings_merged[kings_merged['player'].str.contains('total|team', case=False, na=False)]
kings_vals = kings_team_row.iloc[0] if not kings_team_row.empty else kings_merged.mean(numeric_only=True)

# League totals → per-game
league_vals = league_teams.mean(numeric_only=True)
g_kings = kings_vals.get('g', 82)
g_league = league_vals.get('g', 82)

# PER-GAME
count_cols = ['pts','ast','trb','stl','blk','orb','drb','tov']
for col in count_cols:
    if col in kings_vals: kings_vals[col] = kings_vals[col] / g_kings
    if col in league_vals: league_vals[col] = league_vals[col] / g_league

# SHOOTING %
pct_cols = ['fg_pct','3p_pct','ft_pct']
for col in pct_cols:
    if col not in kings_vals: kings_vals[col] = np.nan
    if col not in league_vals: league_vals[col] = np.nan

# TS%
def calc_ts(row):
    fga = row.get('fga', 1)
    fta = row.get('fta', 1)
    pts = row.get('pts', 1)
    return pts / (2 * (fga + 0.44 * fta)) if (fga + 0.44 * fta) > 0 else 0.5

kings_vals['ts_pct'] = calc_ts(kings_vals)
league_vals['ts_pct'] = calc_ts(league_vals)

# BPM
kings_vals['bpm'] = kings_merged['bpm'].mean()
league_vals['bpm'] = 0.0

# BUILD COMPARISON
comparison = pd.DataFrame({
    'kings': [kings_vals.get(m, np.nan) for m in metrics],
    'league_avg': [max(league_vals.get(m, 0.1), 0.1) for m in metrics]  # FORCE MIN 0.1
}, index=metrics)

comparison['difference'] = comparison['kings'] - comparison['league_avg']
comparison.loc['tov', 'difference'] *= -1

comparison['weight'] = comparison['difference'].abs()
comparison['weight'] /= comparison['weight'].sum()
comparison.loc['bpm', 'weight'] *= 1.8
comparison['weight'] /= comparison['weight'].sum()

print("KINGS WEAKNESSES (SAFE):")
display(comparison.round(3))

KINGS WEAKNESSES (SAFE):


,kings,league_avg,difference,weight
pts,0.15,1.39,-1.24,0.19
ast,0.04,0.32,-0.29,0.04
trb,0.06,0.54,-0.48,0.07
stl,0.01,0.10,-0.09,0.01
blk,0.01,0.10,-0.09,0.01
orb,0.01,0.14,-0.12,0.02
drb,0.04,0.40,-0.36,0.05
tov,0.02,0.17,0.16,0.02
fg_pct,0.47,0.47,-0.00,0.00
3p_pct,0.31,0.36,-0.05,0.01


## Step 7: Map Prospect Stats to Standard Metrics  
I aligned prospect per-game and efficiency stats with the same 11 metrics used for the Kings (e.g., `pts`, `bpm`, `ts_pct`). 

This ensures direct comparability. Any remaining missing values are filled with **column medians** to avoid dropping players.

In [216]:
prospect_final = prospects_agg[[
    'first_name','last_name','team','league','age','games'
]].copy()

# Map raw → standard
prospect_final['pts'] = prospects_agg['points_pg']
prospect_final['ast'] = prospects_agg['assists_pg']
prospect_final['trb'] = prospects_agg['offensive_rebounds_pg'] + prospects_agg['defensive_rebounds_pg']
prospect_final['stl'] = prospects_agg['steals_pg']
prospect_final['blk'] = prospects_agg['blocked_shots_pg']
prospect_final['tov'] = prospects_agg['turnovers_pg']
prospect_final['fg_pct'] = prospects_agg['fg_pct']
prospect_final['3p_pct'] = prospects_agg['3p_pct']
prospect_final['ft_pct'] = prospects_agg['ft_pct']
prospect_final['ts_pct'] = prospects_agg['ts_pct']
prospect_final['bpm'] = prospects_agg['internal_box_plus_minus']
prospect_final['orb'] = prospects_agg['offensive_rebounds_pg']
prospect_final['drb'] = prospects_agg['defensive_rebounds_pg']

# Fill NaN
for col in metrics:
    prospect_final[col] = prospect_final[col].fillna(prospect_final[col].median())

## Step 7.5: Position Inference (Aside, as I did not include in fit score)  
We infer player positions using simple, transparent rules based on per-game stats. This helps scouts visualize **where** a prospect fits in the Kings' lineup.  
- **PG**: High 3P% + High AST  
- **SG**: High 3P% + Scoring  
- **SF**: Balanced scoring + rebounding  
- **PF**: Strong offensive rebounding  
- **C**: High blocks or defensive rebounding  
This is **not used in fit score** — only for display.

In [217]:
def infer_position(row):
    if row['3p_pct'] >= 0.37 and row['ast'] >= 4.5:
        return 'PG'
    elif row['3p_pct'] >= 0.36 and row['pts'] >= 16:
        return 'SG'
    elif row['pts'] >= 15 and row['trb'] >= 6:
        return 'SF'
    elif row['orb'] >= 2.0 and row['drb'] >= 4.5:
        return 'PF'
    elif row['blk'] >= 1.2 or row['drb'] >= 7.0:
        return 'C'
    else:
        return 'Wing'

prospect_final['position'] = prospect_final.apply(infer_position, axis=1)

print("Position Distribution:")
display(prospect_final['position'].value_counts())

Position Distribution:


position
Wing    100
C         3
SF        1
PG        1
Name: count, dtype: int64

## Step 8: Calculate Fit Score Using Kings Needs Only  
We compute a **"Kings Fit Score"** by:  
1. Using **per-game stats** from international prospects.  
2. **Multiplying each stat by its Kings need weight**.  
3. **Inverting TOV** (lower turnovers = higher score).  
4. **Summing into a single fit score**.  

**No division by league averages → no `inf` → no errors.**  
High `fit_score` = player **directly fills Sacramento’s biggest holes**.

In [218]:
for col in key_cols:
    prospects_agg[f'{col}_pg'] = prospects_agg[col] / prospects_agg['games']

# Shooting % (from Step 4)
# prospects_agg has fg_pct, 3p_pct, ft_pct

# Map
prospect_metrics = {
    'pts': prospects_agg['points_pg'],
    'ast': prospects_agg['assists_pg'],
    'trb': prospects_agg['offensive_rebounds_pg'] + prospects_agg['defensive_rebounds_pg'],
    'stl': prospects_agg['steals_pg'],
    'blk': prospects_agg['blocked_shots_pg'],
    'orb': prospects_agg['offensive_rebounds_pg'],
    'drb': prospects_agg['defensive_rebounds_pg'],
    'tov': prospects_agg['turnovers_pg'],
    'fg_pct': prospects_agg['fg_pct'],
    '3p_pct': prospects_agg['3p_pct'],
    'ft_pct': prospects_agg['ft_pct'],
    'ts_pct': prospects_agg['ts_pct'],
    'bpm': prospects_agg['internal_box_plus_minus']
}

prospects_norm = pd.DataFrame(prospect_metrics)

# SAFE NORMALIZATION
for m in metrics:
    league_val = float(comparison.loc[m, 'league_avg'])  # guaranteed > 0.1
    if m == 'tov':
        prospects_norm[m + '_norm'] = league_val / prospects_norm[m].replace(0, 0.1)
    else:
        prospects_norm[m + '_norm'] = prospects_norm[m] / league_val

# WEIGHTED
for m in metrics:
    weight = float(comparison.loc[m, 'weight'])
    prospects_norm[m + '_weighted'] = prospects_norm[m + '_norm'] * weight

# FIT SCORE
prospects_norm['fit_score'] = prospects_norm[[f'{m}_weighted' for m in metrics]].sum(axis=1)

# FINAL TABLE
top_prospects = pd.concat([
    prospects_agg[['first_name','last_name','league','team','age']].reset_index(drop=True),
    prospects_norm[['fit_score'] + metrics]
], axis=1).sort_values('fit_score', ascending=False).head(10)

print("\nTOP 10 PROSPECTS (NO INF):")
display(top_prospects[[
    'first_name','last_name','league','age','pts','ast','trb','bpm','ts_pct','fit_score'
]].round(2))

# FINAL CHECK
print("\nAny inf?", np.isinf(top_prospects['fit_score']).any())  # → False


TOP 10 PROSPECTS (NO INF):


,first_name,last_name,league,age,pts,ast,trb,bpm,ts_pct,fit_score
29,Diaw,Anigbogu,EuroCup,24.00,6.29,0.82,4.06,6.28,0.71,36.14
99,Watson,Ware,Italy - Liga A,21.90,6.00,3.20,2.50,6.06,0.64,34.87
49,Ivica,Lydon,Spain - ACB,24.00,9.27,0.91,5.73,5.62,0.75,33.50
41,Gaffney,Bozeman,Italy - Liga A,24.10,13.11,3.67,6.61,4.76,0.59,30.04
89,Stanislav,Nesby,EuroCup,23.70,5.57,1.57,1.71,5.05,0.72,28.86
78,Rod,Haslem,EuroLeague,22.80,9.08,3.45,4.24,4.79,0.59,28.84
9,Barrett,Stackhouse,EuroCup,18.10,3.93,0.33,2.47,4.93,0.68,28.03
58,Kuzminskas,Ventura,EuroLeague,22.30,12.08,1.15,6.38,4.35,0.63,27.44
37,Farley,Atkins,EuroCup,24.80,15.27,2.82,5.55,4.28,0.64,27.40
91,Theis,Wagner,EuroCup,24.40,12.68,0.90,5.00,3.14,0.69,20.32



Any inf? False


## Step 9: Generate Recommendations Table  
The top 10 prospects are ranked by fit score. A clean, readable table displays:  
- Player name  
- League  
- Age  
- Key stats (`pts`, `ast`, `trb`, `bpm`, `ts_pct`)  
- Final `fit_score`  
This is the **final scouting recommendation list**.

In [219]:
recommendations = top_prospects[[
    'first_name','last_name','league','age',
    'pts','ast','trb','bpm','ts_pct','fit_score'
]].round(2).copy()

recommendations['Player'] = recommendations['first_name'] + ' ' + recommendations['last_name']
recommendations = recommendations[['Player','league','age','pts','ast','trb','bpm','ts_pct','fit_score']]

print("\nTOP 10 PROSPECTS TO SCOUT:")
display(recommendations.reset_index(drop=True))


TOP 10 PROSPECTS TO SCOUT:


,Player,league,age,pts,ast,trb,bpm,ts_pct,fit_score
0,Diaw Anigbogu,EuroCup,24.00,6.29,0.82,4.06,6.28,0.71,36.14
1,Watson Ware,Italy - Liga A,21.90,6.00,3.20,2.50,6.06,0.64,34.87
2,Ivica Lydon,Spain - ACB,24.00,9.27,0.91,5.73,5.62,0.75,33.50
3,Gaffney Bozeman,Italy - Liga A,24.10,13.11,3.67,6.61,4.76,0.59,30.04
4,Stanislav Nesby,EuroCup,23.70,5.57,1.57,1.71,5.05,0.72,28.86
5,Rod Haslem,EuroLeague,22.80,9.08,3.45,4.24,4.79,0.59,28.84
6,Barrett Stackhouse,EuroCup,18.10,3.93,0.33,2.47,4.93,0.68,28.03
7,Kuzminskas Ventura,EuroLeague,22.30,12.08,1.15,6.38,4.35,0.63,27.44
8,Farley Atkins,EuroCup,24.80,15.27,2.82,5.55,4.28,0.64,27.40
9,Theis Wagner,EuroCup,24.40,12.68,0.90,5.00,3.14,0.69,20.32


## Step 10: Save Results to SQLite Database  

I would have loved to create a dashboard, but I did not have time. Wrangling and analyzing the data took longer than expected.

All processed data — full prospect pool, top 10, and recommendations — are saved to `kings_scouting.db` in three tables:  
- `prospects_final`  
- `top_prospects`  
- `recommendations`  
This enables easy access for future analysis or dashboard integration.

In [220]:
conn = sqlite3.connect('kings_scouting.db')
prospect_final.to_sql('prospects_final', conn, if_exists='replace', index=False)
top_prospects.to_sql('top_prospects', conn, if_exists='replace', index=False)
recommendations.to_sql('recommendations', conn, if_exists='replace', index=False)
conn.close()

print("\nSaved to kings_scouting.db")


Saved to kings_scouting.db
